# **Capstone Model**

# **Data Loading, EDA, Understanding**

In [ ]:
DATA_FILE =  "gt-framework-synth.xlsx"

In [ ]:
#import as needed
import pandas as pd
import numpy as np

In [ ]:
sheets = pd.read_excel(DATA_FILE, sheet_name=None)

In [ ]:
sheets

{'DATA_DICTIONARY':                  Sheet name  \
 0              A_brand_kpis   
 1   B_competitive_benchmark   
 2                C_personas   
 3      D_platform_inventory   
 4                 E_reg_fit   
 5                F_bop_risk   
 6     GT_Series_Assumptions   
 7                I_team_tam   
 8    J_competitor_economics   
 9       Synthetic_BOM_Proxy   
 10     Unit_Economics_Model   
 11         O_stage_gate_log   
 
                                               Purpose  \
 0                Brand objectives + weighted KPI tree   
 1   Competitive GT participation/activation benchm...   
 2       Customer racing personas + preference weights   
 3   Candidate production platforms (GM + comparato...   
 4   Platform-to-GT feasibility + change burden + c...   
 5   Performance balancing (BoP-like) risk + adjust...   
 6   Anonymized GT-series tier definitions + constr...   
 7   Market size proxies by series & region (synthe...   
 8            Competitor economics proxie

In [ ]:
sheets


{'DATA_DICTIONARY':                  Sheet name  \
 0              A_brand_kpis   
 1   B_competitive_benchmark   
 2                C_personas   
 3      D_platform_inventory   
 4                 E_reg_fit   
 5                F_bop_risk   
 6     GT_Series_Assumptions   
 7                I_team_tam   
 8    J_competitor_economics   
 9       Synthetic_BOM_Proxy   
 10     Unit_Economics_Model   
 11         O_stage_gate_log   
 
                                               Purpose  \
 0                Brand objectives + weighted KPI tree   
 1   Competitive GT participation/activation benchm...   
 2       Customer racing personas + preference weights   
 3   Candidate production platforms (GM + comparato...   
 4   Platform-to-GT feasibility + change burden + c...   
 5   Performance balancing (BoP-like) risk + adjust...   
 6   Anonymized GT-series tier definitions + constr...   
 7   Market size proxies by series & region (synthe...   
 8            Competitor economics proxie

In [ ]:
df_brand_kpis         = sheets["A_brand_kpis"]
df_competitive        = sheets["B_competitive_benchmark"]
df_personas           = sheets["C_personas"]
df_platforms          = sheets["D_platform_inventory"]
df_reg_fit            = sheets["E_reg_fit"]
df_bop_risk           = sheets["F_bop_risk"]
df_gt_assumptions     = sheets["GT_Series_Assumptions"]
df_team_tam           = sheets["I_team_tam"]
df_competitor_econ    = sheets["J_competitor_economics"]
df_bom                = sheets["Synthetic_BOM_Proxy"]
df_unit_economics     = sheets["Unit_Economics_Model"]
df_stage_gate         = sheets["O_stage_gate_log"]

In [ ]:
print('eda: sizes of each sheet within excel')
for sheet_name, df in sheets.items():
    print('')
    print(f"{sheet_name}: {df.shape[0]} rows × {df.shape[1]} cols")

eda: sizes of each sheet within excel

DATA_DICTIONARY: 12 rows × 4 cols

A_brand_kpis: 25 rows × 10 cols

B_competitive_benchmark: 20 rows × 9 cols

C_personas: 5 rows × 16 cols

D_platform_inventory: 10 rows × 15 cols

E_reg_fit: 20 rows × 12 cols

F_bop_risk: 15 rows × 10 cols

GT_Series_Assumptions: 2 rows × 13 cols

I_team_tam: 18 rows × 8 cols

J_competitor_economics: 20 rows × 9 cols

O_stage_gate_log: 6 rows × 7 cols

Synthetic_BOM_Proxy: 160 rows × 10 cols

Unit_Economics_Model: 20 rows × 11 cols


**Gate 0**

As our final output desire is a go / no go decision model based on multiple data points, we first define what the OEM motors definition of success should be.

In this model success is likely defined by _____.


we want to first explore the historical and projected KPIs to understand performance and benchmarking.


In [ ]:

  #this function will take in the excel file and compute the brand intent from historical data
  kpis  = sheets["A_brand_kpis"]
  bench = sheets["B_competitive_benchmark"]

  pillar_weights = kpis.groupby("brand_pillar")["weight"].sum().sort_values(ascending=False)
  top_pillar     = pillar_weights.idxmax() #gets highest val

In [ ]:
  top_pillar

'Premium Brand Equity'

In [ ]:
#as capstone advisor gave us certain name, we want to turn the brand pillar into the language we want
objective_map = {
        "Performance Credibility":      "Halo performance credibility",
        "Customer Racing Profitability": "Customer racing profitability",
        "Accessible Motorsport":         "Dealer / enthusiast activation",
        "Premium Brand Equity":          "Tiered ladder strategy",
        "Technology Leadership":         "Technology leadership / EV halo",}


oem_objective = objective_map.get(top_pillar, f"Custom objective via '{top_pillar}'")

In [ ]:
  print('From the brand pillar weights, our top objective is:    ',oem_objective)

From the brand pillar weights, our top objective is:     Tiered ladder strategy


In [ ]:
  kpis_sorted = kpis.sort_values("weight", ascending=False)

  top_kpis = kpis_sorted[["kpi", "brand_pillar", "weight", "baseline",
                                "target_year1", "unit"]].head(5)

In [ ]:
  top_kpis

,kpi,brand_pillar,weight,baseline,target_year1,unit
16,Earned Media Volume,Premium Brand Equity,0.243,122.59,137.37,articles/month
4,Earned Media Volume,Performance Credibility,0.236,50.54,59.92,articles/month
17,Dealer Event Attendance,Premium Brand Equity,0.218,171.20,204.02,people/event
22,Qualified Leads,Customer Racing Profitability,0.210,178.58,223.67,leads/month
18,Customer Racing Unit Sales,Premium Brand Equity,0.198,68.66,81.63,cars/year


In [ ]:
#now we calculate brand intent score (we have the expected growth and historica perf and weight)

kpis["lift"] = (kpis["target_year1"] - kpis["baseline"]) / kpis["baseline"].replace(0, np.nan)
#basically (old - new) over old

In [ ]:
kpis["lift"].head(10)

,lift
0,0.282686
1,0.211688
2,0.200269
3,0.070020
4,0.185596
5,0.334848
6,0.089969
7,0.190504
8,0.193189
9,0.144194


In [ ]:
kpis["contrib"] = kpis["lift"] * kpis["weight"]
#multi by weight as in data

#formattng
brand_intent_score = round(kpis["contrib"].sum() * 100, 2)

In [ ]:
brand_intent_score

np.float64(74.1)

In [ ]:
kpis["contrib"].head(10)

,contrib
0,0.039011
1,0.031753
2,0.037250
3,0.007562
4,0.043801
5,0.032145
6,0.013945
7,0.017336
8,0.023762
9,0.023792


In [ ]:
bench

,oem,class_focus,series_presence,activation_tactics,estimated_spend_tier,customer_racing_emphasis,earned_media_index,social_index,notes
0,Porsche,GT3,IMSA GTD; WEC LMGT3,Driver development ladder; Sim racing tie-ins;...,Medium,Medium,43.8,58.2,High-volume customer car sales model
1,Porsche,GT4,SRO GT4 America; GT4 European Series; WEC LMGT3,Dealer track nights; Driver development ladder...,Medium,High,84.7,62.8,Strong dealer activation footprint
2,BMW,GT3,WEC LMGT3,Tech storytelling; Influencer/creator content;...,High,Medium,73.0,82.3,High-volume customer car sales model
3,BMW,GT4,SRO GT4 America; SRO GTWC Europe; IMSA GTD,Hospitality suites; Influencer/creator content...,High,Low,85.2,84.4,Halo-focused endurance presence
4,Ford,GT3,IMSA Pilot Challenge GS,Driver development ladder; Influencer/creator ...,High,Low,40.4,62.3,Halo-focused endurance presence
5,Ford,GT4,SRO GT4 America; IMSA GTD,Customer racing academy; Hospitality suites; T...,High,Medium,65.5,38.6,Factory-backed customer teams
6,Mercedes-AMG,GT3,GT4 European Series; SRO GTWC Europe,Tech storytelling; Hospitality suites; Influen...,High,Medium,69.8,74.5,Factory-backed customer teams
7,Mercedes-AMG,GT4,IMSA GTD,Customer racing academy; Driver development la...,High,Low,85.6,96.2,Strong dealer activation footprint
8,Ferrari,GT3,GT4 European Series; IMSA Pilot Challenge GS,Influencer/creator content; Driver development...,Low,Medium,47.0,50.0,Regional-heavy program
9,Ferrari,GT4,GT4 European Series; WEC LMGT3,Influencer/creator content; Customer racing ac...,High,High,76.3,43.6,Strong dealer activation footprint


In [ ]:
#benchmark from sheet B now that we understand the intent of the brand (its strength) + top pillar of tiered ladder strategy (aka premium brand equity)

avg_earned     = round(bench["earned_media_index"].mean(),2)
avg_social     = round(bench["social_index"].mean(),2)
top_earned_idx = round(bench["earned_media_index"].idxmax(),2)
top_earned_val = round(bench.loc[top_earned_idx, "earned_media_index"],2)

In [ ]:
('avg earned',avg_earned)

('avg earned', np.float64(69.48))

In [ ]:
('avg social',avg_social)

('avg social', np.float64(67.15))

In [ ]:
top_earned_idx

14

In [ ]:
total_pillar_weight = pillar_weights.sum()

In [ ]:
w_earned          = pillar_weights.get("Performance Credibility", 0) / total_pillar_weight
w_social          = pillar_weights.get("Premium Brand Equity", 0) / total_pillar_weight
w_customer_racing = pillar_weights.get("Customer Racing Profitability", 0) / total_pillar_weight


In [ ]:
w_earned

np.float64(0.22138024357239514)

In [ ]:

#normalize and scale datsa as needed
w_total = w_earned + w_social + w_customer_racing
w_earned          /= w_total
w_social          /= w_total
w_customer_racing /= w_total

#calc based on weights
bench["comp_score"] = (
    bench["earned_media_index"] * w_earned +
    bench["social_index"]       * w_social +
    bench["customer_racing_emphasis"].map({"High": 10, "Medium": 5, "Low": 1}) * w_customer_racing)


In [ ]:
print(f"Weights → earned: {w_earned:.2f} | social: {w_social:.2f} | customer_racing: {w_customer_racing:.2f}")

Weights → earned: 0.35 | social: 0.36 | customer_racing: 0.29


In [ ]:
# we assign each criteria of the sheet a weight of importance
bench["comp_score"] = (
    bench["earned_media_index"] * w_earned +
    bench["social_index"]       * w_social +
    bench["customer_racing_emphasis"].map({"High": 10, "Medium": 5, "Low": 1}) * w_customer_racing)

In [ ]:
bench["comp_score"].head(8)

,comp_score
0,37.700902
1,55.166423
2,56.597037
3,60.473851
4,36.815672
5,38.295406
6,52.676642
7,64.844912


In [ ]:
top_competitor = bench.loc[top_earned_idx, "oem"]
top_comp_score = round(bench.loc[top_earned_idx, "comp_score"], 2)

In [ ]:
#depending on most important category, we need leaders for either earned media or social presence

# but we know OEM is a new market entry
leader_earned = bench.loc[bench["earned_media_index"].idxmax(), ["oem","earned_media_index"]]
leader_social  = bench.loc[bench["social_index"].idxmax(),       ["oem","social_index"]]

In [ ]:
leader_earned

,14
oem,McLaren
earned_media_index,93.6


In [ ]:
leader_social

,7
oem,Mercedes-AMG
social_index,96.2


In [ ]:
#it's also helpful to see average values to see competitors overall
peer_threshold = bench["comp_score"].quantile(0.75)
peer_set       = bench[bench["comp_score"] >= peer_threshold]
avg_earned     = round(peer_set["earned_media_index"].mean(), 2)
avg_social     = round(peer_set["social_index"].mean(), 2)

In [ ]:
peer_threshold


np.float64(55.31227994847575)

In [ ]:
peer_set  #these peers are in the top 75% of performers for overall competitor score

,oem,class_focus,series_presence,activation_tactics,estimated_spend_tier,customer_racing_emphasis,earned_media_index,social_index,notes,comp_score
2,BMW,GT3,WEC LMGT3,Tech storytelling; Influencer/creator content;...,High,Medium,73.0,82.3,High-volume customer car sales model,56.597037
3,BMW,GT4,SRO GT4 America; SRO GTWC Europe; IMSA GTD,Hospitality suites; Influencer/creator content...,High,Low,85.2,84.4,Halo-focused endurance presence,60.473851
7,Mercedes-AMG,GT4,IMSA GTD,Customer racing academy; Driver development la...,High,Low,85.6,96.2,Strong dealer activation footprint,64.844912
13,Toyota,GT4,IMSA GTD,Driver development ladder; Customer racing aca...,Medium,Low,69.3,86.8,High-volume customer car sales model,55.749850
19,GM,GT4,IMSA Pilot Challenge GS; GT4 European Series; ...,Driver development ladder; Customer racing aca...,High,Low,76.6,87.5,Factory-backed customer teams,58.564749


In [ ]:
avg_earned

np.float64(77.94)

In [ ]:
avg_social

np.float64(87.44)

In [ ]:
print('mercedes wins on overall composite score')

mercedes wins on overall composite score


In [ ]:
all_tactics = bench["activation_tactics"].dropna().str.split("; ").explode()

In [ ]:
bench["activation_tactics"].value_counts()

#this lists what are currently and historically used)

,count
activation_tactics,
Driver development ladder; Customer racing academy; Sim racing tie-ins,2
Dealer track nights; Driver development ladder; Influencer/creator content,1
Driver development ladder; Sim racing tie-ins; Tech storytelling,1
Tech storytelling; Influencer/creator content; Sim racing tie-ins,1
Hospitality suites; Influencer/creator content; Customer racing academy,1
Customer racing academy; Hospitality suites; Tech storytelling,1
Driver development ladder; Influencer/creator content; Customer racing academy,1
Customer racing academy; Driver development ladder; Tech storytelling,1
Influencer/creator content; Driver development ladder; Dealer track nights,1


In [ ]:
top_tactics = all_tactics.value_counts().head(5).index.tolist()

In [ ]:
top_tactics

['Customer racing academy',
 'Influencer/creator content',
 'Driver development ladder',
 'Tech storytelling',
 'Sim racing tie-ins']

In [ ]:
high_cr = (bench["customer_racing_emphasis"] == "High").sum()

In [ ]:
high_cr #reports how many with high racing emphasis

np.int64(4)

In [ ]:
print("Output for Gate 0: brand intent")
print("*"*55)

print("Our OEM objective:", oem_objective)
print("Our OEM top pillar:", top_pillar)

print('')
print("----pillar weights:")
for pillar, w in pillar_weights.items():
    bar = "█" * int(w * 30)
    print(" ", pillar, round(w,3), bar)

print("\nbrand intent score:", brand_intent_score, "→ GO ✓" if brand_intent_score > 0 else "→ REVIEW ✗")

print("\ntop kpis:")
for _, row in top_kpis.iterrows():
    lift = ((row["target_year1"] - row["baseline"]) / row["baseline"]) * 100
    print(" ", row["kpi"], "| w =", round(row["weight"],3), "| lift:", str(round(lift,1)) + "%")

print("\ncompetitors (n=20)")
print("  avg earned:", avg_earned, "| avg social:", avg_social)
print("  leader:", top_competitor, "| score:", top_earned_val)
print("  high customer racing emphasis:", high_cr, "of", len(bench))
print("  common tactics:", ", ".join(top_tactics))
print("─"*55)

Output for Gate 0: brand intent
*******************************************************
Our OEM objective: Tiered ladder strategy
Our OEM top pillar: Premium Brand Equity

----pillar weights:
  Premium Brand Equity 0.835 █████████████████████████
  Performance Credibility 0.818 ████████████████████████
  Technology Leadership 0.736 ██████████████████████
  Customer Racing Profitability 0.676 ████████████████████
  Accessible Motorsport 0.63 ██████████████████

brand intent score: 74.1 → GO ✓

top kpis:
  Earned Media Volume | w = 0.243 | lift: 12.1%
  Earned Media Volume | w = 0.236 | lift: 18.6%
  Dealer Event Attendance | w = 0.218 | lift: 19.2%
  Qualified Leads | w = 0.21 | lift: 25.2%
  Customer Racing Unit Sales | w = 0.198 | lift: 18.9%

competitors (n=20)
  avg earned: 77.94 | avg social: 87.44
  leader: McLaren | score: 93.6
  high customer racing emphasis: 4 of 20
  common tactics: Customer racing academy, Influencer/creator content, Driver development ladder, Tech storytelli

# **End of Gate 0 here**

# **Start of Gate 1 here**

Question: which production vehicle is plausible?

- given stable platform, public + synthetic data
+ feasibility score

In [ ]:
df_platforms          = sheets["D_platform_inventory"]
df_reg_fit            = sheets["E_reg_fit"]
df_bom                = sheets["Synthetic_BOM_Proxy"]

In [ ]:
df_platforms

,platform_id,oem,brand,nameplate,body_style,layout,powertrain,wheelbase_in,track_width_in,curb_weight_lb,msrp_usd,program_status,global_availability_score,motorsport_heritage_score,platform_stability_score
0,PLT001,GM,Chevrolet,Chevrolet Camaro (Gen6),Coupe,FR,ICE,112.3,70.9,3750,65000,Sunset,3,2,3
1,PLT002,GM,Chevrolet,Chevrolet Corvette (C8),Coupe,MR,ICE,107.2,76.0,3660,80000,Active,2,2,2
2,PLT003,GM,Cadillac,Cadillac CT5-V Blackwing,Sedan,FR,ICE,116.0,74.1,4100,95000,Active,3,2,4
3,PLT004,GM,Chevrolet,Chevrolet Silverado (1500),Truck,FR,ICE,147.5,81.2,5000,55000,Active,4,2,2
4,PLT005,GM,Cadillac,Cadillac Lyriq,SUV,AWD,EV,122.0,77.8,5600,65000,Active,2,3,2
5,PLT006,Porsche,Porsche,Porsche 718 Cayman,Coupe,MR,ICE,97.4,70.9,3200,68000,Active,3,4,2
6,PLT007,Ford,Ford,Ford Mustang (S650),Coupe,FR,ICE,107.0,75.4,3900,45000,Active,3,3,4
7,PLT008,BMW,BMW,BMW M4,Coupe,FR,ICE,112.5,73.6,3830,80000,Active,2,4,4
8,PLT009,Mercedes-AMG,Mercedes-AMG,Mercedes-AMG GT,Coupe,FR,ICE,106.3,76.3,3850,100000,Active,3,2,3
9,PLT010,Toyota,Toyota,Toyota GR Supra,Coupe,FR,ICE,97.2,73.0,3400,56000,Active,4,3,1


In [ ]:
df_platforms.describe()

,wheelbase_in,track_width_in,curb_weight_lb,msrp_usd,global_availability_score,motorsport_heritage_score,platform_stability_score
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.00000
mean,112.540000,74.920000,4029.000000,70900.000000,2.900000,2.700000,2.70000
std,14.477584,3.152706,730.379658,17691.491992,0.737865,0.823273,1.05935
min,97.200000,70.900000,3200.000000,45000.000000,2.000000,2.000000,1.00000
25%,106.475000,73.150000,3682.500000,58250.000000,2.250000,2.000000,2.00000
50%,109.750000,74.750000,3840.000000,66500.000000,3.000000,2.500000,2.50000
75%,115.125000,76.225000,4050.000000,80000.000000,3.000000,3.000000,3.75000
max,147.500000,81.200000,5600.000000,100000.000000,4.000000,4.000000,4.00000


In [ ]:
df_reg_fit.head(5)

,platform_id,nameplate,target_class,feasible_flag,required_changes,packaging_risk,engineering_complexity_score_1_5,estimated_homologation_timeline_months,key_unknowns,gt_series_tier_recommendation,conversion_invasiveness_score_1_5,compliance_risk_index_0_1
0,PLT001,Chevrolet Camaro (Gen6),GT3,Yes,Brakes + ABS calibration; Cooling package; Fue...,Medium,4,25,Aero stability across ride heights,Advanced Tier,4,0.46
1,PLT001,Chevrolet Camaro (Gen6),GT4,Yes,Electronics/CAN overlay; Aero package; Driver ...,High,3,14,Aero stability across ride heights,Entry Tier,3,0.79
2,PLT002,Chevrolet Corvette (C8),GT3,Yes,Brakes + ABS calibration; Driver safety system...,High,2,25,Cooling margin at high ambient,Advanced Tier,4,0.74
3,PLT002,Chevrolet Corvette (C8),GT4,Yes,Aero package; Brakes + ABS calibration; Driver...,High,4,15,Electronic architecture integration,Entry Tier,4,0.81
4,PLT003,Cadillac CT5-V Blackwing,GT3,No,Cooling package; Roll cage integration; Fuel c...,High,4,16,Electronic architecture integration,Advanced Tier,5,0.81


In [ ]:
df_bom.head(4)

,platform_id,nameplate,gt_series_tier,subsystem,estimated_cost_low_usd,estimated_cost_high_usd,lead_time_band,single_source_risk,assembly_complexity_score_1_5,serviceability_score_1_5
0,PLT001,Chevrolet Camaro (Gen6),Entry Tier,Safety cage & seat systems,73233.0,99080.0,16-24 weeks,Low,3.0,5.0
1,PLT001,Chevrolet Camaro (Gen6),Entry Tier,Fuel system & plumbing,61917.0,83770.0,4-8 weeks,Low,3.0,3.0
2,PLT001,Chevrolet Camaro (Gen6),Entry Tier,Brakes/ABS & hydraulics,26249.0,35514.0,4-8 weeks,Low,4.0,3.0
3,PLT001,Chevrolet Camaro (Gen6),Entry Tier,Suspension & dampers,29886.0,40434.0,2-4 weeks,Medium,4.0,3.0


In [ ]:
#we take average of each of the 3 scores to make a combined col metric
df_platforms["avg_plat_score"] = df_platforms[["global_availability_score","motorsport_heritage_score","platform_stability_score"]].mean(axis=1)


In [ ]:
df_platforms["avg_plat_score"]

,avg_plat_score
0,2.666667
1,2.000000
2,3.000000
3,2.666667
4,2.333333
5,3.000000
6,3.333333
7,3.333333
8,2.666667
9,2.666667


In [ ]:

merged_reg_plat = df_reg_fit.merge(df_platforms[["platform_id","avg_plat_score","program_status"]], on="platform_id", how="left")

In [ ]:
merged_reg_plat

,platform_id,nameplate,target_class,feasible_flag,required_changes,packaging_risk,engineering_complexity_score_1_5,estimated_homologation_timeline_months,key_unknowns,gt_series_tier_recommendation,conversion_invasiveness_score_1_5,compliance_risk_index_0_1,avg_plat_score,program_status
0,PLT001,Chevrolet Camaro (Gen6),GT3,Yes,Brakes + ABS calibration; Cooling package; Fue...,Medium,4,25,Aero stability across ride heights,Advanced Tier,4,0.46,2.666667,Sunset
1,PLT001,Chevrolet Camaro (Gen6),GT4,Yes,Electronics/CAN overlay; Aero package; Driver ...,High,3,14,Aero stability across ride heights,Entry Tier,3,0.79,2.666667,Sunset
2,PLT002,Chevrolet Corvette (C8),GT3,Yes,Brakes + ABS calibration; Driver safety system...,High,2,25,Cooling margin at high ambient,Advanced Tier,4,0.74,2.000000,Active
3,PLT002,Chevrolet Corvette (C8),GT4,Yes,Aero package; Brakes + ABS calibration; Driver...,High,4,15,Electronic architecture integration,Entry Tier,4,0.81,2.000000,Active
4,PLT003,Cadillac CT5-V Blackwing,GT3,No,Cooling package; Roll cage integration; Fuel c...,High,4,16,Electronic architecture integration,Advanced Tier,5,0.81,3.000000,Active
5,PLT003,Cadillac CT5-V Blackwing,GT4,Yes,Driver safety systems; Electronics/CAN overlay...,High,2,11,Cooling margin at high ambient,Entry Tier,4,0.79,3.000000,Active
6,PLT004,Chevrolet Silverado (1500),GT3,No,Roll cage integration; Aero package; Electroni...,High,4,15,Aero stability across ride heights,Advanced Tier,5,0.67,2.666667,Active
7,PLT004,Chevrolet Silverado (1500),GT4,No,Brakes + ABS calibration; Aero package; Fuel c...,High,4,13,Electronic architecture integration,Entry Tier,4,0.77,2.666667,Active
8,PLT005,Cadillac Lyriq,GT3,No,Suspension + dampers; Driver safety systems; F...,High,4,15,Weight distribution and ballast window,Advanced Tier,4,0.80,2.333333,Active
9,PLT005,Cadillac Lyriq,GT4,No,Aero package; Driver safety systems; Fuel cell...,High,4,16,Aero stability across ride heights,Entry Tier,5,0.81,2.333333,Active


In [ ]:
#we set criteria of when to kill project

merged_reg_plat["kill"] = ((merged_reg_plat["feasible_flag"] != "Yes") | (merged_reg_plat["program_status"] == "Sunset") |
        (merged_reg_plat["compliance_risk_index_0_1"] > 0.75))

In [ ]:
merged_reg_plat['program_status'].value_counts()

,count
program_status,
Active,18
Sunset,2


In [ ]:
def get_kill_reason(row):
    kill_reason = ""
    if row["feasible_flag"] != "Yes":
        kill_reason += "not feasible; "

    if row["program_status"] == "Sunset":
        kill_reason += "platform sunset; "

    if row["compliance_risk_index_0_1"] > 0.75:
        kill_reason += "we found compliance risk too high; "

    return kill_reason.strip() if kill_reason else "None"

In [ ]:
merged_reg_plat["kill_reasons_verbose"] = merged_reg_plat.apply(get_kill_reason, axis=1)

In [ ]:
merged_reg_plat["kill_reasons_verbose"]

,kill_reasons_verbose
0,platform sunset;
1,platform sunset; we found compliance risk too ...
2,None
3,we found compliance risk too high;
4,not feasible; we found compliance risk too high;
5,we found compliance risk too high;
6,not feasible;
7,not feasible; we found compliance risk too high;
8,not feasible; we found compliance risk too high;
9,not feasible; we found compliance risk too high;


In [ ]:
bom_agg = df_bom.groupby(["platform_id","gt_series_tier"])["estimated_cost_high_usd"].sum().reset_index()
bom_agg.columns = ["platform_id","gt_series_tier","total_bom_cost"]


In [ ]:
bom_agg

,platform_id,gt_series_tier,total_bom_cost
0,PLT001,Advanced Tier,641264.0
1,PLT001,Entry Tier,451371.0
2,PLT002,Advanced Tier,673990.0
3,PLT002,Entry Tier,505243.0
4,PLT003,Advanced Tier,698717.0
5,PLT003,Entry Tier,583049.0
6,PLT004,Advanced Tier,0.0
7,PLT004,Entry Tier,0.0
8,PLT005,Advanced Tier,0.0
9,PLT005,Entry Tier,0.0


In [ ]:
# platform feasibility score (higher = better, 0-10)
merged_reg_plat["feasibility_score"] = (merged_reg_plat["avg_plat_score"] / 5) * 10 * (1 - merged_reg_plat["compliance_risk_index_0_1"])
merged_reg_plat.loc[merged_reg_plat["kill"], "feasibility_score"] = 0

In [ ]:
merged_reg_plat

,platform_id,nameplate,target_class,feasible_flag,required_changes,packaging_risk,engineering_complexity_score_1_5,estimated_homologation_timeline_months,key_unknowns,gt_series_tier_recommendation,conversion_invasiveness_score_1_5,compliance_risk_index_0_1,avg_plat_score,program_status,kill,kill_reasons_verbose,feasibility_score
0,PLT001,Chevrolet Camaro (Gen6),GT3,Yes,Brakes + ABS calibration; Cooling package; Fue...,Medium,4,25,Aero stability across ride heights,Advanced Tier,4,0.46,2.666667,Sunset,True,platform sunset;,0.000000
1,PLT001,Chevrolet Camaro (Gen6),GT4,Yes,Electronics/CAN overlay; Aero package; Driver ...,High,3,14,Aero stability across ride heights,Entry Tier,3,0.79,2.666667,Sunset,True,platform sunset; we found compliance risk too ...,0.000000
2,PLT002,Chevrolet Corvette (C8),GT3,Yes,Brakes + ABS calibration; Driver safety system...,High,2,25,Cooling margin at high ambient,Advanced Tier,4,0.74,2.000000,Active,False,None,1.040000
3,PLT002,Chevrolet Corvette (C8),GT4,Yes,Aero package; Brakes + ABS calibration; Driver...,High,4,15,Electronic architecture integration,Entry Tier,4,0.81,2.000000,Active,True,we found compliance risk too high;,0.000000
4,PLT003,Cadillac CT5-V Blackwing,GT3,No,Cooling package; Roll cage integration; Fuel c...,High,4,16,Electronic architecture integration,Advanced Tier,5,0.81,3.000000,Active,True,not feasible; we found compliance risk too high;,0.000000
5,PLT003,Cadillac CT5-V Blackwing,GT4,Yes,Driver safety systems; Electronics/CAN overlay...,High,2,11,Cooling margin at high ambient,Entry Tier,4,0.79,3.000000,Active,True,we found compliance risk too high;,0.000000
6,PLT004,Chevrolet Silverado (1500),GT3,No,Roll cage integration; Aero package; Electroni...,High,4,15,Aero stability across ride heights,Advanced Tier,5,0.67,2.666667,Active,True,not feasible;,0.000000
7,PLT004,Chevrolet Silverado (1500),GT4,No,Brakes + ABS calibration; Aero package; Fuel c...,High,4,13,Electronic architecture integration,Entry Tier,4,0.77,2.666667,Active,True,not feasible; we found compliance risk too high;,0.000000
8,PLT005,Cadillac Lyriq,GT3,No,Suspension + dampers; Driver safety systems; F...,High,4,15,Weight distribution and ballast window,Advanced Tier,4,0.80,2.333333,Active,True,not feasible; we found compliance risk too high;,0.000000
9,PLT005,Cadillac Lyriq,GT4,No,Aero package; Driver safety systems; Fuel cell...,High,4,16,Aero stability across ride heights,Entry Tier,5,0.81,2.333333,Active,True,not feasible; we found compliance risk too high;,0.000000


In [ ]:
shortlist = merged_reg_plat[~merged_reg_plat["kill"]][["nameplate","target_class","gt_series_tier_recommendation",
                                "feasibility_score","estimated_homologation_timeline_months"]].copy()

In [ ]:
shortlist

,nameplate,target_class,gt_series_tier_recommendation,feasibility_score,estimated_homologation_timeline_months
2,Chevrolet Corvette (C8),GT3,Advanced Tier,1.040000,25
10,Porsche 718 Cayman,GT3,Advanced Tier,1.680000,16
12,Ford Mustang (S650),GT3,Advanced Tier,5.333333,20
14,BMW M4,GT3,Advanced Tier,4.666667,17
15,BMW M4,GT4,Entry Tier,5.266667,13
16,Mercedes-AMG GT,GT3,Advanced Tier,2.400000,24
17,Mercedes-AMG GT,GT4,Entry Tier,3.786667,15
18,Toyota GR Supra,GT3,Advanced Tier,2.560000,23


In [ ]:
print("\ngate 1 — platform filter")
print(merged_reg_plat[["nameplate","target_class","kill","kill_reasons_verbose","feasibility_score"]].to_string(index=False))


gate 1 — platform filter
                 nameplate target_class  kill                                kill_reasons_verbose  feasibility_score
   Chevrolet Camaro (Gen6)          GT3  True                                    platform sunset;           0.000000
   Chevrolet Camaro (Gen6)          GT4  True platform sunset; we found compliance risk too high;           0.000000
   Chevrolet Corvette (C8)          GT3 False                                                None           1.040000
   Chevrolet Corvette (C8)          GT4  True                  we found compliance risk too high;           0.000000
  Cadillac CT5-V Blackwing          GT3  True    not feasible; we found compliance risk too high;           0.000000
  Cadillac CT5-V Blackwing          GT4  True                  we found compliance risk too high;           0.000000
Chevrolet Silverado (1500)          GT3  True                                       not feasible;           0.000000
Chevrolet Silverado (1500)          GT

In [ ]:
print("\nshortlist —", len(shortlist), "combos made it")
print(shortlist.to_string(index=False))


shortlist — 8 combos made it
              nameplate target_class gt_series_tier_recommendation  feasibility_score  estimated_homologation_timeline_months
Chevrolet Corvette (C8)          GT3                 Advanced Tier           1.040000                                      25
     Porsche 718 Cayman          GT3                 Advanced Tier           1.680000                                      16
    Ford Mustang (S650)          GT3                 Advanced Tier           5.333333                                      20
                 BMW M4          GT3                 Advanced Tier           4.666667                                      17
                 BMW M4          GT4                    Entry Tier           5.266667                                      13
        Mercedes-AMG GT          GT3                 Advanced Tier           2.400000                                      24
        Mercedes-AMG GT          GT4                    Entry Tier           3.786667   

# **Gate 2**

De-risk homologation and performance balancing (Can it meet requirements and compete sustainably?)

In [ ]:
bop = sheets["F_bop_risk"]
reg = sheets["E_reg_fit"]
assumptions = sheets["GT_Series_Assumptions"]

In [ ]:
bop

,platform_id,nameplate,target_class,bop_volatility_index_0_1,setup_window_score_1_5,drivability_score_1_5,analog_competitors,mitigation_notes,gt_series_tier_recommendation,balancing_adjustment_frequency_band
0,PLT001,Chevrolet Camaro (Gen6),GT3,0.78,2,3,Cadillac Lyriq; Cadillac CT5-V Blackwing,Ensure ballast window via lightweight components,Advanced Tier,High
1,PLT001,Chevrolet Camaro (Gen6),GT4,0.38,2,4,Chevrolet Silverado (1500); Chevrolet Corvette...,Ensure ballast window via lightweight components,Entry Tier,Medium
2,PLT002,Chevrolet Corvette (C8),GT3,0.54,4,3,BMW M4; Cadillac CT5-V Blackwing,Focus on tire energy management and cooling ro...,Advanced Tier,Medium
3,PLT002,Chevrolet Corvette (C8),GT4,0.39,4,2,Cadillac CT5-V Blackwing; Ford Mustang (S650),Ensure ballast window via lightweight components,Entry Tier,Medium
4,PLT003,Cadillac CT5-V Blackwing,GT4,0.19,4,3,Porsche 718 Cayman; Ford Mustang (S650),Focus on tire energy management and cooling ro...,Entry Tier,Low
5,PLT006,Porsche 718 Cayman,GT3,0.40,4,2,Porsche 718 Cayman; Toyota GR Supra,Design aero adjustability; prioritize stable b...,Advanced Tier,Medium
6,PLT006,Porsche 718 Cayman,GT4,0.77,3,2,Porsche 718 Cayman; Chevrolet Camaro (Gen6),Build wide setup window with conservative damp...,Entry Tier,High
7,PLT007,Ford Mustang (S650),GT3,0.19,2,4,Mercedes-AMG GT; BMW M4,Ensure ballast window via lightweight components,Advanced Tier,Low
8,PLT007,Ford Mustang (S650),GT4,0.43,2,2,Chevrolet Camaro (Gen6); Ford Mustang (S650),Build wide setup window with conservative damp...,Entry Tier,Medium
9,PLT008,BMW M4,GT3,0.78,2,4,Mercedes-AMG GT; Cadillac Lyriq,Design aero adjustability; prioritize stable b...,Advanced Tier,High


In [ ]:
reg

,platform_id,nameplate,target_class,feasible_flag,required_changes,packaging_risk,engineering_complexity_score_1_5,estimated_homologation_timeline_months,key_unknowns,gt_series_tier_recommendation,conversion_invasiveness_score_1_5,compliance_risk_index_0_1
0,PLT001,Chevrolet Camaro (Gen6),GT3,Yes,Brakes + ABS calibration; Cooling package; Fue...,Medium,4,25,Aero stability across ride heights,Advanced Tier,4,0.46
1,PLT001,Chevrolet Camaro (Gen6),GT4,Yes,Electronics/CAN overlay; Aero package; Driver ...,High,3,14,Aero stability across ride heights,Entry Tier,3,0.79
2,PLT002,Chevrolet Corvette (C8),GT3,Yes,Brakes + ABS calibration; Driver safety system...,High,2,25,Cooling margin at high ambient,Advanced Tier,4,0.74
3,PLT002,Chevrolet Corvette (C8),GT4,Yes,Aero package; Brakes + ABS calibration; Driver...,High,4,15,Electronic architecture integration,Entry Tier,4,0.81
4,PLT003,Cadillac CT5-V Blackwing,GT3,No,Cooling package; Roll cage integration; Fuel c...,High,4,16,Electronic architecture integration,Advanced Tier,5,0.81
5,PLT003,Cadillac CT5-V Blackwing,GT4,Yes,Driver safety systems; Electronics/CAN overlay...,High,2,11,Cooling margin at high ambient,Entry Tier,4,0.79
6,PLT004,Chevrolet Silverado (1500),GT3,No,Roll cage integration; Aero package; Electroni...,High,4,15,Aero stability across ride heights,Advanced Tier,5,0.67
7,PLT004,Chevrolet Silverado (1500),GT4,No,Brakes + ABS calibration; Aero package; Fuel c...,High,4,13,Electronic architecture integration,Entry Tier,4,0.77
8,PLT005,Cadillac Lyriq,GT3,No,Suspension + dampers; Driver safety systems; F...,High,4,15,Weight distribution and ballast window,Advanced Tier,4,0.80
9,PLT005,Cadillac Lyriq,GT4,No,Aero package; Driver safety systems; Fuel cell...,High,4,16,Aero stability across ride heights,Entry Tier,5,0.81


In [ ]:
assumptions

,gt_series_tier,typical_vehicle_basis,homologation_complexity,typical_homologation_timeline_months_range,conversion_invasiveness_score_range_1_5,annual_operating_cost_usd_band,rebuild_interval_hours_range,performance_balancing_mechanism,balancing_volatility_index_range_0_1,serviceability_priority,customer_support_expectation,data_policy_default,notes
0,Entry Tier,Production-derived; cost-contained customer ra...,Low-Medium,10-18,2-4,180k-650k,50-90,Generic balancing adjustments (weight/power/aero),0.15-0.55,Very High,High parts availability; predictable updates; ...,Public results/timing only; no OEM confidentia...,Modeled on typical GT4-like conditions; values...
1,Advanced Tier,Production-derived GT; higher performance and ...,Medium-High,14-26,3-5,650k-2.2M,35-70,Generic balancing adjustments (weight/power/aero),0.25-0.85,High,Strong spares network; professional support; t...,Public results/timing + optional customer data...,Modeled on typical GT3/LMGT3-like conditions; ...


In [ ]:
gate2_merged = shortlist.merge(
    bop[["platform_id","nameplate","target_class","bop_volatility_index_0_1",
         "setup_window_score_1_5","drivability_score_1_5","mitigation_notes"]],
    left_on=["nameplate","target_class"],
    right_on=["nameplate","target_class"],
    how="left")

In [ ]:
gate2_merged

,nameplate,target_class,gt_series_tier_recommendation,feasibility_score,estimated_homologation_timeline_months,platform_id,bop_volatility_index_0_1,setup_window_score_1_5,drivability_score_1_5,mitigation_notes
0,Chevrolet Corvette (C8),GT3,Advanced Tier,1.040000,25,PLT002,0.54,4,3,Focus on tire energy management and cooling ro...
1,Porsche 718 Cayman,GT3,Advanced Tier,1.680000,16,PLT006,0.40,4,2,Design aero adjustability; prioritize stable b...
2,Ford Mustang (S650),GT3,Advanced Tier,5.333333,20,PLT007,0.19,2,4,Ensure ballast window via lightweight components
3,BMW M4,GT3,Advanced Tier,4.666667,17,PLT008,0.78,2,4,Design aero adjustability; prioritize stable b...
4,BMW M4,GT4,Entry Tier,5.266667,13,PLT008,0.24,4,3,Design aero adjustability; prioritize stable b...
5,Mercedes-AMG GT,GT3,Advanced Tier,2.400000,24,PLT009,0.18,4,3,Build wide setup window with conservative damp...
6,Mercedes-AMG GT,GT4,Entry Tier,3.786667,15,PLT009,0.66,3,2,Build wide setup window with conservative damp...
7,Toyota GR Supra,GT3,Advanced Tier,2.560000,23,PLT010,0.22,2,2,Build wide setup window with conservative damp...


In [ ]:
gate2_merged = gate2_merged.merge(
    reg[["nameplate","target_class","compliance_risk_index_0_1","engineering_complexity_score_1_5",
         "estimated_homologation_timeline_months"]],
    on=["nameplate","target_class"],
    how="left"
)

In [ ]:
gate2_merged

,nameplate,target_class,gt_series_tier_recommendation,feasibility_score,estimated_homologation_timeline_months_x,platform_id,bop_volatility_index_0_1,setup_window_score_1_5,drivability_score_1_5,mitigation_notes,compliance_risk_index_0_1,engineering_complexity_score_1_5,estimated_homologation_timeline_months_y
0,Chevrolet Corvette (C8),GT3,Advanced Tier,1.040000,25,PLT002,0.54,4,3,Focus on tire energy management and cooling ro...,0.74,2,25
1,Porsche 718 Cayman,GT3,Advanced Tier,1.680000,16,PLT006,0.40,4,2,Design aero adjustability; prioritize stable b...,0.72,3,16
2,Ford Mustang (S650),GT3,Advanced Tier,5.333333,20,PLT007,0.19,2,4,Ensure ballast window via lightweight components,0.20,4,20
3,BMW M4,GT3,Advanced Tier,4.666667,17,PLT008,0.78,2,4,Design aero adjustability; prioritize stable b...,0.30,3,17
4,BMW M4,GT4,Entry Tier,5.266667,13,PLT008,0.24,4,3,Design aero adjustability; prioritize stable b...,0.21,4,13
5,Mercedes-AMG GT,GT3,Advanced Tier,2.400000,24,PLT009,0.18,4,3,Build wide setup window with conservative damp...,0.55,4,24
6,Mercedes-AMG GT,GT4,Entry Tier,3.786667,15,PLT009,0.66,3,2,Build wide setup window with conservative damp...,0.29,2,15
7,Toyota GR Supra,GT3,Advanced Tier,2.560000,23,PLT010,0.22,2,2,Build wide setup window with conservative damp...,0.52,2,23


In [ ]:
gate2_merged["combined_risk"] = (gate2_merged["bop_volatility_index_0_1"] * 5 +
    gate2_merged["compliance_risk_index_0_1"] * 5).round(2)

In [ ]:
gate2_merged

,nameplate,target_class,gt_series_tier_recommendation,feasibility_score,estimated_homologation_timeline_months_x,platform_id,bop_volatility_index_0_1,setup_window_score_1_5,drivability_score_1_5,mitigation_notes,compliance_risk_index_0_1,engineering_complexity_score_1_5,estimated_homologation_timeline_months_y,combined_risk
0,Chevrolet Corvette (C8),GT3,Advanced Tier,1.040000,25,PLT002,0.54,4,3,Focus on tire energy management and cooling ro...,0.74,2,25,6.40
1,Porsche 718 Cayman,GT3,Advanced Tier,1.680000,16,PLT006,0.40,4,2,Design aero adjustability; prioritize stable b...,0.72,3,16,5.60
2,Ford Mustang (S650),GT3,Advanced Tier,5.333333,20,PLT007,0.19,2,4,Ensure ballast window via lightweight components,0.20,4,20,1.95
3,BMW M4,GT3,Advanced Tier,4.666667,17,PLT008,0.78,2,4,Design aero adjustability; prioritize stable b...,0.30,3,17,5.40
4,BMW M4,GT4,Entry Tier,5.266667,13,PLT008,0.24,4,3,Design aero adjustability; prioritize stable b...,0.21,4,13,2.25
5,Mercedes-AMG GT,GT3,Advanced Tier,2.400000,24,PLT009,0.18,4,3,Build wide setup window with conservative damp...,0.55,4,24,3.65
6,Mercedes-AMG GT,GT4,Entry Tier,3.786667,15,PLT009,0.66,3,2,Build wide setup window with conservative damp...,0.29,2,15,4.75
7,Toyota GR Supra,GT3,Advanced Tier,2.560000,23,PLT010,0.22,2,2,Build wide setup window with conservative damp...,0.52,2,23,3.70


In [ ]:
gate2_merged["tier_rec"] = gate2_merged["gt_series_tier_recommendation"]
gate2_merged

,nameplate,target_class,gt_series_tier_recommendation,feasibility_score,estimated_homologation_timeline_months_x,platform_id,bop_volatility_index_0_1,setup_window_score_1_5,drivability_score_1_5,mitigation_notes,compliance_risk_index_0_1,engineering_complexity_score_1_5,estimated_homologation_timeline_months_y,combined_risk,tier_rec
0,Chevrolet Corvette (C8),GT3,Advanced Tier,1.040000,25,PLT002,0.54,4,3,Focus on tire energy management and cooling ro...,0.74,2,25,6.40,Advanced Tier
1,Porsche 718 Cayman,GT3,Advanced Tier,1.680000,16,PLT006,0.40,4,2,Design aero adjustability; prioritize stable b...,0.72,3,16,5.60,Advanced Tier
2,Ford Mustang (S650),GT3,Advanced Tier,5.333333,20,PLT007,0.19,2,4,Ensure ballast window via lightweight components,0.20,4,20,1.95,Advanced Tier
3,BMW M4,GT3,Advanced Tier,4.666667,17,PLT008,0.78,2,4,Design aero adjustability; prioritize stable b...,0.30,3,17,5.40,Advanced Tier
4,BMW M4,GT4,Entry Tier,5.266667,13,PLT008,0.24,4,3,Design aero adjustability; prioritize stable b...,0.21,4,13,2.25,Entry Tier
5,Mercedes-AMG GT,GT3,Advanced Tier,2.400000,24,PLT009,0.18,4,3,Build wide setup window with conservative damp...,0.55,4,24,3.65,Advanced Tier
6,Mercedes-AMG GT,GT4,Entry Tier,3.786667,15,PLT009,0.66,3,2,Build wide setup window with conservative damp...,0.29,2,15,4.75,Entry Tier
7,Toyota GR Supra,GT3,Advanced Tier,2.560000,23,PLT010,0.22,2,2,Build wide setup window with conservative damp...,0.52,2,23,3.70,Advanced Tier


In [ ]:
gate2_merged["risk_flag"] = gate2_merged["bop_volatility_index_0_1"] > 0.55

In [ ]:
gate2_merged["risk_flag"]

,risk_flag
0,False
1,False
2,False
3,True
4,False
5,False
6,True
7,False


In [ ]:
cols = ["nameplate","target_class","combined_risk","bop_volatility_index_0_1",
        "compliance_risk_index_0_1","tier_rec","risk_flag","mitigation_notes"]

In [ ]:
print("Output of gate 2 — homologation + bop risk")
print(gate2_merged[cols].to_string(index=False))

Output of gate 2 — homologation + bop risk
              nameplate target_class  combined_risk  bop_volatility_index_0_1  compliance_risk_index_0_1      tier_rec  risk_flag                                           mitigation_notes
Chevrolet Corvette (C8)          GT3           6.40                      0.54                       0.74 Advanced Tier      False     Focus on tire energy management and cooling robustness
     Porsche 718 Cayman          GT3           5.60                      0.40                       0.72 Advanced Tier      False       Design aero adjustability; prioritize stable balance
    Ford Mustang (S650)          GT3           1.95                      0.19                       0.20 Advanced Tier      False           Ensure ballast window via lightweight components
                 BMW M4          GT3           5.40                      0.78                       0.30 Advanced Tier       True       Design aero adjustability; prioritize stable balance
            

In [ ]:
best = gate2_merged.sort_values("combined_risk").iloc[0]


best #output for top of each category

,2
nameplate,Ford Mustang (S650)
target_class,GT3
gt_series_tier_recommendation,Advanced Tier
feasibility_score,5.333333
estimated_homologation_timeline_months_x,20
platform_id,PLT007
bop_volatility_index_0_1,0.19
setup_window_score_1_5,2
drivability_score_1_5,4
mitigation_notes,Ensure ballast window via lightweight components


In [ ]:
print("\nlowest risk combo:", best["nameplate"], "/", best["target_class"], "| risk =", best["combined_risk"])


lowest risk combo: Ford Mustang (S650) / GT3 | risk = 1.95


# **Gate 3**

Prove customer economics (Is this a viable customer racing product?)

Is there enough demand and a credible customer base?
Can the OEM price competitively and still make money on cars + spares + support?
Is the operating cost profile acceptable for the tier?

In [ ]:

df_personas           = sheets["C_personas"]
df_team_tam           = sheets["I_team_tam"]
df_competitor_econ    = sheets["J_competitor_economics"]
df_unit_economics     = sheets["Unit_Economics_Model"]

In [ ]:
df_personas

,persona,segment,gt4_annual_budget_low,gt4_annual_budget_high,gt3_annual_budget_low,gt3_annual_budget_high,primary_regions,primary_series,weight_reliability,weight_parts_availability,weight_purchase_price,weight_operating_cost,weight_bop_stability,weight_drivability,weight_brand_prestige,weight_tech_telemetry
0,Gentleman Driver,Pro-Am,250000,750000,350000,1200000,Global; Europe,WEC LMGT3; IMSA GTD,0.228,0.051,0.010,0.095,0.472,0.013,0.002,0.128
1,Team Owner,Customer Team,500000,1500000,900000,3500000,North America; Europe,SRO GT4 America; SRO GTWC Europe,0.020,0.049,0.007,0.226,0.362,0.169,0.044,0.122
2,Dealer Principal,Channel Partner,0,0,0,0,Europe; Asia-Pacific,GT4 European Series; SRO GTWC Europe,0.064,0.203,0.352,0.020,0.001,0.128,0.188,0.044
3,Pro Driver,Factory/Contract,150000,500000,250000,900000,Global; Asia-Pacific,SRO GTWC Europe; WEC LMGT3,0.203,0.276,0.129,0.283,0.003,0.044,0.045,0.018
4,Track-Day Convert,Aspirational,150000,400000,250000,800000,North America; Global,IMSA GTD; IMSA Pilot Challenge GS,0.434,0.006,0.201,0.013,0.080,0.097,0.036,0.132


In [ ]:
df_team_tam

,series,region,active_teams_est,active_cars_est,new_car_purchases_per_season_est,annual_churn_rate_est,avg_team_budget_band,notes
0,SRO GT4 America,North America,22,29,2,0.11,500k-1.5M,Stable grids
1,SRO GT4 America,Europe,53,83,10,0.08,500k-1.5M,Stable grids
2,SRO GT4 America,Asia-Pacific,64,71,8,0.16,250-750k,High operating costs
3,GT4 European Series,North America,59,89,12,0.14,150-400k,Pro-am heavy
4,GT4 European Series,Europe,57,87,14,0.18,150-400k,Stable grids
5,GT4 European Series,Asia-Pacific,22,27,3,0.15,500k-1.5M,Stable grids
6,IMSA Pilot Challenge GS,North America,57,69,10,0.13,250-750k,Stable grids
7,IMSA Pilot Challenge GS,Europe,54,64,8,0.09,500k-1.5M,Stable grids
8,IMSA Pilot Challenge GS,Asia-Pacific,27,42,4,0.21,900k-3.5M,Stable grids
9,SRO GTWC Europe,North America,29,32,5,0.09,500k-1.5M,Strong manufacturer diversity


In [ ]:
df_competitor_econ

,class,manufacturer,model,racecar_msrp_usd_est,annual_operating_cost_usd_band,parts_basket_index,typical_rebuild_interval_hours,support_model,resale_value_3yr_pct_est
0,GT4,Porsche,718 Cayman GT4 RS Clubsport,238986,250-500k,97.0,65,Regional distributor,0.54
1,GT4,BMW,M4 GT4 Evo,268728,300-650k,83.7,50,Manufacturer + partners,0.57
2,GT4,Ford,Mustang GT4,251209,250-500k,92.5,87,Partner network,0.49
3,GT4,Mercedes-AMG,AMG GT4,285560,180-350k,102.2,61,Partner network,0.57
4,GT4,McLaren,Artura GT4,317113,180-350k,87.1,78,Manufacturer + partners,0.66
5,GT4,Toyota,GR Supra GT4 EVO2,259265,180-350k,119.5,75,Regional distributor,0.58
6,GT4,Aston Martin,Vantage AMR GT4 Evo,292916,300-650k,91.2,78,Regional distributor,0.52
7,GT4,Audi,R8 LMS GT4,313084,180-350k,97.1,59,Regional distributor,0.56
8,GT4,Lotus,Emira GT4,310034,250-500k,110.4,69,Manufacturer + partners,0.54
9,GT4,Ginetta,G56 GT4 Evo,260743,250-500k,119.1,54,Manufacturer + partners,0.67


In [ ]:
tam_total_cars = df_team_tam["active_cars_est"].sum()

In [ ]:
tam_total_cars

np.int64(862)

In [ ]:
tam_new_buys = df_team_tam["new_car_purchases_per_season_est"].sum()

In [ ]:
tam_new_buys

np.int64(108)

In [ ]:
avg_comp_price = df_competitor_econ["racecar_msrp_usd_est"].mean()

In [ ]:
avg_comp_price

np.float64(479403.55)

In [ ]:
df_personas["gt4_mid"] = (df_personas["gt4_annual_budget_low"] + df_personas["gt4_annual_budget_high"]) / 2
df_personas["gt3_mid"] = (df_personas["gt3_annual_budget_low"] + df_personas["gt3_annual_budget_high"]) / 2


In [ ]:
# unit economics — feasible only
ue_summary = df_unit_economics[df_unit_economics["applicable_flag"] == "Yes"].copy()
ue_summary["annual_profit_est"] = (
    ue_summary["gross_margin_pct_est"] * ue_summary["racecar_msrp_usd_est"] +
    ue_summary["annual_spares_revenue_usd_est"] -
    ue_summary["annual_support_cost_usd_est"])

In [ ]:
capture_rate = 0.10
ue_summary["3yr_units"] = tam_new_buys * capture_rate * 3
ue_summary["3yr_rev_est"] = ue_summary["3yr_units"] * ue_summary["racecar_msrp_usd_est"]
ue_summary["3yr_gp_est"] = ue_summary["3yr_units"] * ue_summary["racecar_msrp_usd_est"] * ue_summary["gross_margin_pct_est"]


In [ ]:
ue_summary["econ_score"] = np.where(
    ue_summary["3yr_units"] >= ue_summary["breakeven_units_est"],
    10,
    (ue_summary["3yr_units"] / ue_summary["breakeven_units_est"]) * 10).round(2)

In [ ]:
best_region = df_team_tam.sort_values("active_cars_est", ascending=False).iloc[0]["region"]


In [ ]:
best_region

'North America'

In [ ]:
print("OUTPUT: gate 3! — customer economics")
print("addressable cars:", tam_total_cars, "| new buys/season:", tam_new_buys)
print("avg competitor price:", round(avg_comp_price, 2), "| best region:", best_region)



OUTPUT: gate 3! — customer economics
addressable cars: 862 | new buys/season: 108
avg competitor price: 479403.55 | best region: North America


In [ ]:
print("\nunit economics:")
print(ue_summary[["nameplate","gt_series_tier","annual_profit_est","econ_score"]].to_string(index=False))


unit economics:
               nameplate gt_series_tier  annual_profit_est  econ_score
 Chevrolet Camaro (Gen6)     Entry Tier           90243.95        1.60
 Chevrolet Camaro (Gen6)  Advanced Tier          139429.90        0.93
 Chevrolet Corvette (C8)     Entry Tier           77340.05        1.68
 Chevrolet Corvette (C8)  Advanced Tier          156925.70        2.28
Cadillac CT5-V Blackwing     Entry Tier           82142.55        1.15
Cadillac CT5-V Blackwing  Advanced Tier          187440.40        1.06
      Porsche 718 Cayman     Entry Tier           74890.55        1.38
      Porsche 718 Cayman  Advanced Tier          144139.80        1.17
     Ford Mustang (S650)     Entry Tier          112957.85        1.37
     Ford Mustang (S650)  Advanced Tier          136207.45        2.04
                  BMW M4     Entry Tier           85548.65        1.45
                  BMW M4  Advanced Tier          177175.70        1.37
         Mercedes-AMG GT     Entry Tier          105584.30  

# **GATE 4**

Gate 4 — Confirm brand ladder (Does this build a pipeline and ecosystem?)

Decision questions
Does the GT program support a ladder between tiers and reinforce the OEM’s performance strategy?
Does it activate customers/dealers and support retention/upgrade behavior?

Public + synthetic data used
Personas + ladder assumptions (C_personas)
Competitive activation patterns and strategy benchmarks (B_competitive_benchmark)
KPI alignment back to intent (A_brand_kpis)
Outputs
Brand Ladder Score + ladder plan (touchpoints and KPIs)

In [ ]:
df_brand_kpis         = sheets["A_brand_kpis"]
df_competitive        = sheets["B_competitive_benchmark"]
df_personas           = sheets["C_personas"]

In [ ]:
df_personas

,persona,segment,gt4_annual_budget_low,gt4_annual_budget_high,gt3_annual_budget_low,gt3_annual_budget_high,primary_regions,primary_series,weight_reliability,weight_parts_availability,weight_purchase_price,weight_operating_cost,weight_bop_stability,weight_drivability,weight_brand_prestige,weight_tech_telemetry,gt4_mid,gt3_mid
0,Gentleman Driver,Pro-Am,250000,750000,350000,1200000,Global; Europe,WEC LMGT3; IMSA GTD,0.228,0.051,0.010,0.095,0.472,0.013,0.002,0.128,500000.0,775000.0
1,Team Owner,Customer Team,500000,1500000,900000,3500000,North America; Europe,SRO GT4 America; SRO GTWC Europe,0.020,0.049,0.007,0.226,0.362,0.169,0.044,0.122,1000000.0,2200000.0
2,Dealer Principal,Channel Partner,0,0,0,0,Europe; Asia-Pacific,GT4 European Series; SRO GTWC Europe,0.064,0.203,0.352,0.020,0.001,0.128,0.188,0.044,0.0,0.0
3,Pro Driver,Factory/Contract,150000,500000,250000,900000,Global; Asia-Pacific,SRO GTWC Europe; WEC LMGT3,0.203,0.276,0.129,0.283,0.003,0.044,0.045,0.018,325000.0,575000.0
4,Track-Day Convert,Aspirational,150000,400000,250000,800000,North America; Global,IMSA GTD; IMSA Pilot Challenge GS,0.434,0.006,0.201,0.013,0.080,0.097,0.036,0.132,275000.0,525000.0


In [ ]:
df_personas.value_counts()

,,,,,,,,,,,,,,,,,,count
persona,segment,gt4_annual_budget_low,gt4_annual_budget_high,gt3_annual_budget_low,gt3_annual_budget_high,primary_regions,primary_series,weight_reliability,weight_parts_availability,weight_purchase_price,weight_operating_cost,weight_bop_stability,weight_drivability,weight_brand_prestige,weight_tech_telemetry,gt4_mid,gt3_mid,
Dealer Principal,Channel Partner,0,0,0,0,Europe; Asia-Pacific,GT4 European Series; SRO GTWC Europe,0.064,0.203,0.352,0.020,0.001,0.128,0.188,0.044,0.0,0.0,1
Gentleman Driver,Pro-Am,250000,750000,350000,1200000,Global; Europe,WEC LMGT3; IMSA GTD,0.228,0.051,0.010,0.095,0.472,0.013,0.002,0.128,500000.0,775000.0,1
Pro Driver,Factory/Contract,150000,500000,250000,900000,Global; Asia-Pacific,SRO GTWC Europe; WEC LMGT3,0.203,0.276,0.129,0.283,0.003,0.044,0.045,0.018,325000.0,575000.0,1
Team Owner,Customer Team,500000,1500000,900000,3500000,North America; Europe,SRO GT4 America; SRO GTWC Europe,0.020,0.049,0.007,0.226,0.362,0.169,0.044,0.122,1000000.0,2200000.0,1
Track-Day Convert,Aspirational,150000,400000,250000,800000,North America; Global,IMSA GTD; IMSA Pilot Challenge GS,0.434,0.006,0.201,0.013,0.080,0.097,0.036,0.132,275000.0,525000.0,1


In [ ]:
df_personas["has_ladder"] = (df_personas["gt4_mid"] > 0) & (df_personas["gt3_mid"] > 0)
ladder_coverage = df_personas["has_ladder"].mean()

In [ ]:
ladder_coverage

np.float64(0.8)

In [ ]:
df_personas["has_ladder"] #ladder aka whether they have upgrade feasibility

,has_ladder
0,True
1,True
2,False
3,True
4,True


In [ ]:
bench["active_score"] = (bench["earned_media_index"] + bench["social_index"]) / 2
avg_activation = bench["active_score"].mean()

In [ ]:
bench["active_score"]

,active_score
0,51.00
1,73.75
2,77.65
3,84.80
4,51.35
5,52.05
6,72.15
7,90.90
8,48.50
9,59.95


In [ ]:
kpis["hits_target"] = kpis["target_year2"] > kpis["baseline"]
kpi_alignment = kpis["hits_target"].mean()

In [ ]:
kpis["hits_target"]

,hits_target
0,True
1,True
2,True
3,True
4,True
5,True
6,True
7,True
8,True
9,True


In [ ]:
kpi_alignment

np.float64(1.0)

In [ ]:
ladder_score = round((ladder_coverage * 4 + (avg_activation/100) * 3 + kpi_alignment * 3), 2)


In [ ]:
print("OUTPUT: ----- our brand ladder")
print('')
print("ladder coverage:", round(ladder_coverage, 2), "| avg activation:", round(avg_activation, 2), "| kpi alignment:", round(kpi_alignment, 2))
print("ladder score:", ladder_score, "/ 10")
print("anchor pillar:", top_pillar)

OUTPUT: ----- our brand ladder

ladder coverage: 0.8 | avg activation: 68.31 | kpi alignment: 1.0
ladder score: 8.25 / 10
anchor pillar: Premium Brand Equity


In [ ]:
print("\npersonas:")

#prints persona and want race they are likely to compete in
for _, row in df_personas.iterrows():
    print(" ", row["persona"], "|", row["primary_series"])


personas:
  Gentleman Driver | WEC LMGT3; IMSA GTD
  Team Owner | SRO GT4 America; SRO GTWC Europe
  Dealer Principal | GT4 European Series; SRO GTWC Europe
  Pro Driver | SRO GTWC Europe; WEC LMGT3
  Track-Day Convert | IMSA GTD; IMSA Pilot Challenge GS


# **Gate 5 — Lock platform/tier and execute (What exactly is the commitment?)**


Decision questions
Which platform + tier should leadership approve?
What conditions must be met to proceed (timelines, partners, support readiness)?
Public + synthetic data used
Stage-gate decision log template tying evidence to decisions (O_stage_gate_log)
Weighted scorecard across all gate outputs

In [ ]:
log = sheets["O_stage_gate_log"]

In [ ]:
log

,program,gate,decision,decision_date,primary_evidence_ids,top_risks,conditions_to_proceed
0,OEM GT Entry (synthetic),Intent,Go,2026-01-15,C_personas; A_brand_kpis,BOP volatility higher than desired; Supplier l...,NaN
1,OEM GT Entry (synthetic),Platform Filter,Pivot,2026-02-14,C_personas; A_brand_kpis,BOP volatility higher than desired; Support he...,NaN
2,OEM GT Entry (synthetic),Homologation/BOP,No-Go,2026-03-16,D_platform_inventory; F_bop_risk,Supplier lead-time risk for long-lead componen...,NaN
3,OEM GT Entry (synthetic),Customer Economics,Pivot,2026-04-15,C_personas; B_competitive_benchmark,Supplier lead-time risk for long-lead componen...,NaN
4,OEM GT Entry (synthetic),Brand Ladder,Go,2026-05-15,E_reg_fit; C_personas,Supplier lead-time risk for long-lead componen...,NaN
5,OEM GT Entry (synthetic),Program Lock,No-Go,2026-06-14,D_platform_inventory; A_brand_kpis,Supplier lead-time risk for long-lead componen...,NaN


In [ ]:
brand_score = min(brand_intent_score, 10)

In [ ]:
brand_score

10

In [ ]:
#sorts all by platform econ score then pulls out number (best one is first, index at 0 )

econ_score  = ue_summary.sort_values("econ_score", ascending=False).iloc[0]["econ_score"]
best_econ   = ue_summary.sort_values("econ_score", ascending=False).iloc[0]

In [ ]:
econ_score

np.float64(3.09)

In [ ]:
best_econ

,16
platform_id,PLT009
nameplate,Mercedes-AMG GT
gt_series_tier,Entry Tier
applicable_flag,Yes
racecar_msrp_usd_est,291822.0
cogs_usd_est,248048.0
gross_margin_pct_est,0.15
annual_spares_revenue_usd_est,104919.0
annual_support_cost_usd_est,43108.0
breakeven_units_est,105.0


In [ ]:
best_risk   = gate2_merged.sort_values("combined_risk").iloc[0]
risk_score  = round(max(0, min(10, 10 - best_risk["combined_risk"])), 2)

In [ ]:
best_risk

,2
nameplate,Ford Mustang (S650)
target_class,GT3
gt_series_tier_recommendation,Advanced Tier
feasibility_score,5.333333
estimated_homologation_timeline_months_x,20
platform_id,PLT007
bop_volatility_index_0_1,0.19
setup_window_score_1_5,2
drivability_score_1_5,4
mitigation_notes,Ensure ballast window via lightweight components


In [ ]:
best_plat   = shortlist.sort_values("feasibility_score", ascending=False).iloc[0]
plat_score  = min(best_plat["feasibility_score"], 10)

In [ ]:
best_plat

,12
nameplate,Ford Mustang (S650)
target_class,GT3
gt_series_tier_recommendation,Advanced Tier
feasibility_score,5.333333
estimated_homologation_timeline_months,20


In [ ]:
weights = {"brand": 0.20, "platform": 0.20, "risk": 0.20, "econ": 0.25, "ladder": 0.15}
# we can tweak the above depending on how much we rank each factor


In [ ]:
scores  = {"brand": brand_score, "platform": plat_score, "risk": risk_score,
           "econ": econ_score, "ladder": ladder_score}

In [ ]:
total = round(sum(weights[k] * scores[k] for k in weights), 2)

In [ ]:
total

np.float64(6.69)

In [ ]:
# output for the go no go framework!!
if total >= 7:
    decision = "GO"
elif total >= 5:
    decision = "GO-WITH-CONDITIONS"
else:
    decision = "NO-GO"

In [ ]:
print("OUTPUT: gate 5!")



for k in weights:
    print(" ", k, "| score:", round(scores[k],2), "| weight:", weights[k], "| contrib:", round(weights[k]*scores[k],2))


OUTPUT: gate 5!
  brand | score: 10 | weight: 0.2 | contrib: 2.0
  platform | score: 5.33 | weight: 0.2 | contrib: 1.07
  risk | score: 8.05 | weight: 0.2 | contrib: 1.61
  econ | score: 3.09 | weight: 0.25 | contrib: 0.77
  ladder | score: 8.25 | weight: 0.15 | contrib: 1.24


In [ ]:
print("\ntotal score:", total)
print("recommended platform:", best_plat["nameplate"], "/", best_plat["target_class"])
print("launch region:", best_region)
print("\ndecision:", decision)


total score: 6.69
recommended platform: Ford Mustang (S650) / GT3
launch region: North America

decision: GO-WITH-CONDITIONS


In [ ]:
if decision == "GO-WITH-CONDITIONS":
    print("\nconditions:")
    print("  - confirm homologation timeline with sanctioning body")
    print("  - lock supply chain for long-lead BOM parts")
    print("  - get 2+ anchor team commitments before proceeding")


conditions:
  - confirm homologation timeline with sanctioning body
  - lock supply chain for long-lead BOM parts
  - get 2+ anchor team commitments before proceeding


In [ ]:
# log it
new_row = {
    "program": "GT Stage-Gate Model Output",
    "gate": "Final (Gate 5)",
    "decision": decision,
    "decision_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "primary_evidence_ids": "All gates",
    "top_risks": "BOP risk on " + best_risk["nameplate"] + "; Breakeven=" + str(best_econ["breakeven_units_est"]) + " units",
    "conditions_to_proceed": "see above" if "CONDITIONS" in decision else "None"}

In [ ]:
updated_log = pd.concat([log, pd.DataFrame([new_row])], ignore_index=True)

In [ ]:
updated_log

,program,gate,decision,decision_date,primary_evidence_ids,top_risks,conditions_to_proceed
0,OEM GT Entry (synthetic),Intent,Go,2026-01-15,C_personas; A_brand_kpis,BOP volatility higher than desired; Supplier l...,NaN
1,OEM GT Entry (synthetic),Platform Filter,Pivot,2026-02-14,C_personas; A_brand_kpis,BOP volatility higher than desired; Support he...,NaN
2,OEM GT Entry (synthetic),Homologation/BOP,No-Go,2026-03-16,D_platform_inventory; F_bop_risk,Supplier lead-time risk for long-lead componen...,NaN
3,OEM GT Entry (synthetic),Customer Economics,Pivot,2026-04-15,C_personas; B_competitive_benchmark,Supplier lead-time risk for long-lead componen...,NaN
4,OEM GT Entry (synthetic),Brand Ladder,Go,2026-05-15,E_reg_fit; C_personas,Supplier lead-time risk for long-lead componen...,NaN
5,OEM GT Entry (synthetic),Program Lock,No-Go,2026-06-14,D_platform_inventory; A_brand_kpis,Supplier lead-time risk for long-lead componen...,NaN
6,GT Stage-Gate Model Output,Final (Gate 5),GO-WITH-CONDITIONS,2026-03-24,All gates,BOP risk on Ford Mustang (S650); Breakeven=105...,see above
